# Phase 4 - Engine failure-mode head (NEW in v3)  —  Rotax_912_ULS

Misfire, injector fouling, cooling degradation, combustion instability - PS section C.

These are **engine** faults, not sensor faults. v2 had no such head: those words
existed only in `advisory.py` as labels mapped onto sensor channels.

In [1]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, tensorflow as tf
from tensorflow import keras
import tf_data_pipeline as P, model_architectures as A, train_common as C

C.report_gpu()

  TF 2.16.2  devices ['CPU', 'GPU']


['CPU', 'GPU']

In [2]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----
ENGINE = "Rotax_912_ULS"
steps = C.describe(ENGINE)
train_ds, val_ds, test_ds = C.datasets(ENGINE)

Rotax_912_ULS  (912)
  physics   : v3   TBO 2000.0 h
  rows      : 10,012,939 in 1144 scenarios
  steps/epoch: train 986  val 100  test 122
  window 128  stride 64  features 25  aux 10


2026-09-13 20:47:21.396603: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-13 20:47:21.396634: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-13 20:47:21.396677: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-13 20:47:21.396694: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-13 20:47:21.396705: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


### Why sigmoid, not softmax
Several modes can be present at once, so these are independent severities rather
than a one-of-N choice. That is the structural difference from the diagnosis head.

In [3]:
xi  = A.build_encoder_input()
enc = A.build_encoder(xi)
fm  = A.build_failure_mode_head(enc)
model = keras.Model(xi, {"y_failure_mode": fm}, name="failure_modes")
model.summary()

Model: "failure_modes"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 25)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv1 (Conv1D) │ (None, 128, 48)   │      3,648 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop1          │ (None, 128, 48)   │          0 │ tcn0_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop2          │ (None, 128, 48)   │          0 │ tcn0_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_proj (Conv1D)  │ (None, 128, 48)   │      1,248 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_add (Add)      │ (None, 128, 48)   │          0 │ tcn0_drop2[0][0], │
│                     │                   │            │ tcn0_proj[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn0_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop1          │ (None, 128, 48)   │          0 │ tcn1_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop2          │ (None, 128, 48)   │          0 │ tcn1_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_drop2[0][0], │
│                     │                   │            │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop1          │ (None, 128, 48)   │          0 │ tcn2_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn2_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop2          │ (None, 128, 48)   │          0 │ tcn2_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn2_drop2[0][0], │
│                     │                   │            │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn2_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 91,284 (356.58 KB)

 Trainable params: 91,284 (356.58 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# (nothing extra for this phase)

In [5]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss={"y_failure_mode": keras.losses.BinaryCrossentropy()},
              metrics={"y_failure_mode": [keras.metrics.MeanAbsoluteError(name="mae")]
                                           + C.failure_mode_metrics()})

In [ ]:
PHASE = "phase4_failure_modes"
PREV  = "phase3_severity"
ckpt  = C.ckpt_path(ENGINE, PHASE)
if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))

hist = model.fit(
    train_ds.map(C.split_labels(["y_failure_mode"])),
    validation_data=val_ds.map(C.split_labels(["y_failure_mode"])),
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    epochs=20,
    callbacks=C.callbacks(ckpt, monitor="val_auc_all_det", mode="max"),
    verbose=1,
)
print("saved:", ckpt)

  warm-started 13 layers from phase3_severity_912.keras
  existing checkpoint backed up -> /Users/harsh/Documents/UAV_Engine/validation/models/phase4_failure_modes_912.prev.keras
Epoch 1/20


/Users/harsh/Documents/UAV_Engine/validation/venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-09-13 20:47:23.638683: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


986/986 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - auc_all: 0.7992 - auc_all_det: 0.8534 - auc_combustion_instability: 0.8239 - auc_cooling_degradation: 0.6473 - auc_det_combustion_instability: 0.8649 - auc_det_cooling_degradation: 0.6734 - auc_det_injector_fouling: 0.9027 - auc_det_misfire: 0.9123 - auc_injector_fouling: 0.8183 - auc_misfire: 0.8756 - loss: 0.0691 - mae: 0.0227
Epoch 1: val_auc_all_det improved from None to 0.81381, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase4_failure_modes_912.keras

Epoch 1: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase4_failure_modes_912.keras
986/986 ━━━━━━━━━━━━━━━━━━━━ 222s 219ms/step - auc_all: 0.7992 - auc_all_det: 0.8534 - auc_combustion_instability: 0.8239 - auc_cooling_degradation: 0.6473 - auc_det_combustion_instability: 0.8649 - auc_det_cooling_degradation: 0.6734 - auc_det_injector_fouling: 0.9027 - auc_det_misfire: 0.9123 - auc_injector_fouling: 0.8183 - auc_misfire: 0.8756 - 

In [ ]:
res = model.evaluate(test_ds.map(C.split_labels(["y_failure_mode"])), steps=steps["test"], verbose=1, return_dict=True)
print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))
with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:
    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)